In [1]:
import cadquery as cq
import math
from jupyter_cadquery import *

from jupyter_cadquery.replay import replay, enable_replay, disable_replay

versions()

Overwriting auto display for cadquery Workplane and Shape

Versions:
- jupyter_cadquery  4.0.2
- cad_viewer_widget 3.0.2
- open cascade      7.7.2.1



In [2]:
cv = open_viewer("Windows", cad_width=800, height=600)
set_default_viewer("Windows")

In [3]:
configuration = "6/6"
operation = "single-hung"
window_species = "pine"
size = "12x18"
profile = "ovolo"
thickness = 1.375
stile_width = 2
bottom_rail_width = 3
top_rail_width = 2
muntin_width = 0.75
meeting_rail_width = 1.375
meeting_rail_thickness = 1.75
bay_width = 42
chair_rail_height = 30
configuration_list = configuration.split("/")
size_list = size.split("x")
top_sash_lights = int(configuration_list[0])
bottom_sash_lights = int(configuration_list[1])
light_width = int(size_list[0])
light_height = int(size_list[1])
glazing_rabbet = 0.25
tenon_type = "through"
sash_color = cq.Color(0.95,0.96,0.94,1)

In [4]:
# Precalculate part lengths
configuration_list = configuration.split("/")
size_list = size.split("x")
top_sash_lights = int(configuration_list[0])
bottom_sash_lights = int(configuration_list[1])
light_width = int(size_list[0])
light_height = int(size_list[1])
glazing_rabbet = 0.25
top_stile_length = ((top_sash_lights / 3) * (light_height - (glazing_rabbet * 2))) + top_rail_width + meeting_rail_width + (((top_sash_lights / 3) - 1) * muntin_width)
bottom_stile_length = ((bottom_sash_lights / 3) * (light_height - (glazing_rabbet * 2))) + bottom_rail_width + meeting_rail_width + (((bottom_sash_lights / 3) - 1) * muntin_width)
pulley_stile_length = top_stile_length + bottom_stile_length - meeting_rail_width

if tenon_type == "blind":
    tenon_adjustment = -1
    tenon_length = stile_width + (tenon_adjustment/2)
else:
    tenon_adjustment = 0
    tenon_length = stile_width

rail_length = (light_width * 3) + (stile_width * 2) + (muntin_width * 2) - (glazing_rabbet * 6) + tenon_adjustment
net_rail_length = rail_length - (stile_width * 2)

print("Top Stile: ", top_stile_length, " x ", rail_length)
print("Bottom Stile: ", bottom_stile_length, " x ", rail_length)

Top Stile:  39.125  x  40.0
Bottom Stile:  40.125  x  40.0


In [5]:
def ogee_profile(begin_x, begin_y, direction):

    profile_points = []
    segments = 32
    increment = 90/segments

    # Define the ogee
    stem_width = 0.375
    stem_height = 0.0625
    nose_width = 0.01
    stile_rabbet = 0.4375 #1-3/8 - 15/16
    glazing_rabbet = 0.25
    ogee_diameter = (1.375 - (stem_width + stile_rabbet + stem_height))
    ogee_radius = ogee_diameter / 2
    vertical_radius = glazing_rabbet / 2

    # Add the concave ogee points
    concave_center_x = -begin_x + stile_rabbet + stem_width + ogee_radius # begin x = thickness
    concave_center_y = begin_y # begin y = rail_width

    # Generate concave ogee points
    for segment in range(1,segments):

        if direction == -1:
            angle_degrees = 360 - (segment * increment)
        else:
            angle_degrees = 180 + (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = concave_center_x + (ogee_radius * math.cos(angle_radians))
        arc_y = concave_center_y + (vertical_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Add the convex ogee points
    convex_center_x = concave_center_x
    convex_center_y = concave_center_y - (2 * vertical_radius)

    for segment in range(1, segments):

        if direction == -1:
            angle_degrees = 90 + (segment * increment)
        else:
            angle_degrees = 90 - (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = convex_center_x + (ogee_radius * math.cos(angle_radians))
        arc_y = convex_center_y + (vertical_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    return profile_points

In [6]:
def add_rail(rail_width, rail_length):

    profile_points = []
    segments = 32
    increment = 90/segments

    # Define the ogee
    stem_width = 0.375
    stem_height = 0.0625
    nose_width = 0.01
    stile_rabbet = 0.4375 #1-3/8 - 15/16
    ovolo_radius = (1.375 - (stem_width + stile_rabbet + stem_height))
    ogee_radius = ovolo_radius/2
    vertical_radius = glazing_rabbet / 2

    # Append the initial points
    profile_points.append((0,0))
    profile_points.append((-thickness,0))
    profile_points.append((-thickness,rail_width - glazing_rabbet))
    profile_points.append((-thickness + stile_rabbet, rail_width - glazing_rabbet))
    profile_points.append((-thickness + stile_rabbet, rail_width))
    profile_points.append((-thickness + stile_rabbet + stem_width, rail_width))

    # Calculate the ogee
    ogee = ogee_profile(thickness, rail_width,1)
    profile_points += ogee

    # Add the final points
    profile_points.append((-stem_height, rail_width - (vertical_radius * 2)))
    profile_points.append((0, rail_width - (vertical_radius * 2)))

    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    # Create the 3D stile
    rail = profile .extrude(rail_length)

    return rail

In [7]:
def add_stile(stile_length):

    profile_points = []
    segments = 32
    increment = 90/segments

    # Define the ogee
    stem_width = 0.375
    stem_height = 0.0625
    nose_width = 0.01
    stile_rabbet = 0.4375 #1-3/8 - 15/16
    ovolo_radius = (1.375 - (stem_width + stile_rabbet + stem_height))
    ogee_radius = ovolo_radius/2
    vertical_radius = glazing_rabbet / 2


    # Append the initial points
    profile_points.append((0,0))
    profile_points.append((-thickness,0))
    profile_points.append((-thickness,stile_width - glazing_rabbet))
    profile_points.append((-thickness + stile_rabbet, stile_width - glazing_rabbet))
    profile_points.append((-thickness + stile_rabbet, stile_width))
    profile_points.append((-thickness + stile_rabbet + stem_width, stile_width))
    #profile_points.append((-thickness + stile_rabbet + stem_width, stile_width - stem_height))

    # Calculate the ogee
    ogee = ogee_profile(thickness, 2, 1)
    profile_points += ogee

    # Add the final points
    profile_points.append((-stem_height, stile_width - (vertical_radius * 2)))
    profile_points.append((0, stile_width - (vertical_radius * 2)))

    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    # Create the 3D stile
    stile = profile.extrude(stile_length)

    return stile

In [8]:
def beaded_board(width, height, bead_size):

    profile_points = []
    segments = 32
    increment = 180/segments

    # Define the bead
    bead_diameter = bead_size
    bead_radius = bead_diameter / 2
    board_height = height - bead_diameter
    board_width = width
    center_x = board_width - bead_radius
    center_y = -height + bead_radius

    # Add initial point
    profile_points.append((0,0))
    profile_points.append((board_width,0))
    profile_points.append((board_width, -board_height))

    # Add the bead points from 90 to 270 degrees
    for segment in range(1,segments):

        if segment <= (segments/2):
            angle_degrees = 90 - (segment * increment)
        else:
            segment_counter = segment - (segments/2)
            angle_degrees = 360 - (segment_counter * increment)

        angle_radians = math.radians(angle_degrees)
        
        bead_x = center_x + (bead_radius * math.cos(angle_radians))
        bead_y = center_y + (bead_radius * math.sin(angle_radians))

        profile_points.append((bead_x, bead_y))

    # Add the board corners
    profile_points.append((0, -height))

    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile

In [9]:
def beaded_sill(width, inside_height, outside_height, bead_size):

    profile_points = []
    segments = 32
    increment = 180/segments

    # Define the bead
    bead_diameter = bead_size
    bead_radius = bead_diameter / 2
    board_height = inside_height - bead_diameter
    board_width = width
    center_x = board_width - bead_radius
    center_y = -inside_height + bead_radius
    rain_stem = 0.5
    rain_slope = (inside_height - outside_height)
    siding_notch = 0.375
    wall_width = 4.00
    siding_notch_x = wall_width + siding_notch

    # Add initial point
    profile_points.append((0,0))
    profile_points.append((rain_stem, 0))
    profile_points.append((rain_stem, -rain_stem))
    profile_points.append((board_width, -rain_slope))
    profile_points.append((board_width, -board_height))

    # Add the bead points from 90 to 270 degrees
    for segment in range(1,segments):

        if segment <= (segments/2):
            angle_degrees = 90 - (segment * increment)
        else:
            segment_counter = segment - (segments/2)
            angle_degrees = 360 - (segment_counter * increment)

        angle_radians = math.radians(angle_degrees)
        
        bead_x = center_x + (bead_radius * math.cos(angle_radians))
        bead_y = center_y + (bead_radius * math.sin(angle_radians))

        profile_points.append((bead_x, bead_y))

    # Add the board corners
    profile_points.append((siding_notch_x, -inside_height))
    profile_points.append((siding_notch_x, -inside_height + siding_notch))
    profile_points.append((wall_width, -inside_height + siding_notch))
    profile_points.append((wall_width, -inside_height))
    profile_points.append((0, -inside_height))

    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile

In [10]:
def add_top_meeting_rail(rail_length):

    profile_points = []
    stile_rabbet = 0.4375
    stile_bevel = 0.625
    flat_area = 0.75
    bevel = 0.25

    # Define the ogee
    stem_width = 0.375
    stem_height = 0.0625
    nose_width = 0.01
    stile_rabbet = 0.4375 #1-3/8 - 15/16
    ovolo_radius = (1.375 - (stem_width + stile_rabbet + stem_height))
    ogee_radius = ovolo_radius/2
    vertical_radius = glazing_rabbet / 2
    thickness_diff = (meeting_rail_thickness - thickness)

    # Append the initial points
    #profile_points.append((0, ovolo_radius))
    profile_points.append((0,meeting_rail_width))
    profile_points.append((0, bevel))
    profile_points.append((stile_rabbet, bevel))
    profile_points.append((stile_rabbet, 0))
    profile_points.append((meeting_rail_thickness - bevel, 0))
    profile_points.append((meeting_rail_thickness - bevel, meeting_rail_width - stile_bevel))
    profile_points.append((meeting_rail_thickness, meeting_rail_width - stile_rabbet))
    profile_points.append((meeting_rail_thickness, meeting_rail_width))

    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close().extrude(rail_length)

    return profile

In [11]:
def add_bottom_meeting_rail(rail_length):

    profile_points = []
    stile_rabbet = 0.4375
    stile_bevel = 0.625
    flat_area = 0.75
    bevel = 0.25

    # Define the ogee
    stem_width = 0.375
    stem_height = 0.0625
    nose_width = 0.01
    stile_rabbet = 0.4375 #1-3/8 - 15/16
    ovolo_radius = (1.375 - (stem_width + stile_rabbet + stem_height))
    ogee_radius = ovolo_radius/2
    vertical_radius = glazing_rabbet / 2
    thickness_diff = (meeting_rail_thickness - thickness)

    # Append the initial points
    #profile_points.append((0, ovolo_radius))
    profile_points.append((0,meeting_rail_width))

    ogee = ogee_profile(0.8125, ogee_radius + stem_height,1)
    profile_points += ogee

    profile_points.append((0.500, 0))
    profile_points.append((0.500 + stem_width, 0))
    profile_points.append((0.500 + stem_width, 0.25))
    profile_points.append((0.500 + stem_width + 0.1875, 0.25))
    profile_points.append((0.500 + stem_width + 0.1875, 0))
    profile_points.append((meeting_rail_thickness - bevel, 0))
    profile_points.append((meeting_rail_thickness - bevel, meeting_rail_width - stile_bevel))
    profile_points.append((meeting_rail_thickness, meeting_rail_width - stile_rabbet))
    profile_points.append((meeting_rail_thickness, meeting_rail_width))

    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close().extrude(rail_length)

    return profile

In [12]:
def ovolo_muntin(
    width: float = 1.25,
    height: float = 1.0,
    nose_width: float = 0.25,
    nose_height: float = 0.125,
    wing_width: float = 0.125,
    wing_height: float = 0.25,
    stem_width: float = 0.5,
    stem_height: float = 0.25
) -> cq.Workplane():

    segments = 32
    increment = 90/segments
    ovolo_radius = width/2 - ((nose_width/2) + wing_width)
    print("ovolo radius for muntin: ", ovolo_radius)
    profile_points = []
    bottom_width = (width - stem_width)/2

    # Left wing
    profile_points.append((0,-wing_height))
    profile_points.append((0,0))
    profile_points.append((wing_width,0))

    # Left arc parameters
    center_x = width/2 - (nose_width/2)
    center_y = 0
    
    # Left arc
    for segment in range(1,segments):

        angle_degrees = 180 - (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = center_x + (ovolo_radius * math.cos(angle_radians))
        arc_y = center_y + (ovolo_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Nose
    profile_points.append((width/2 - nose_width/2, ovolo_radius))
    profile_points.append((width/2 - nose_width/2, ovolo_radius + nose_height))
    profile_points.append((width/2 + nose_width/2, ovolo_radius + nose_height))
    profile_points.append((width/2 + nose_width/2, ovolo_radius))

    # Right arc parameters
    center_x = width/2 + (nose_width/2)
    center_y = 0
    
    # Right arc
    for segment in range(1,segments):

        angle_degrees = 90 - (segment * increment)
        angle_radians = math.radians(angle_degrees)

        arc_x = center_x + (ovolo_radius * math.cos(angle_radians))
        arc_y = center_y + (ovolo_radius * math.sin(angle_radians))

        profile_points.append((arc_x, arc_y))

    # Right wing
    profile_points.append((width - wing_width, 0))
    profile_points.append((width, 0))
    profile_points.append((width, -wing_height))

    # Stem
    profile_points.append((width - bottom_width, -wing_height))
    profile_points.append((width - bottom_width, -wing_height - stem_height))
    profile_points.append((bottom_width, -wing_height - stem_height))
    profile_points.append((bottom_width, -wing_height))

    # Close the loop
    profile_points.append((0,-wing_height))
    
    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close()

    return profile
    

In [13]:
def ogee_muntin(length):

    profile_points = []

    profile_points.append((0.625,-0.25))
    profile_points.append((0.625,0))

    # Calculate the ogee
    ogee = ogee_profile(0, 0, 1)
    profile_points += ogee
    list_length = len(profile_points) - 1
    ogee_x = profile_points[list_length][0]
    ogee_y = profile_points[list_length][1]
    profile_points.append((ogee_x + 0.0625, ogee_y))
    profile_points.append((ogee_x + 0.0625, ogee_y - 0.25))
    profile_points.append((ogee_x, ogee_y - 0.25))

    reverse_ogee = ogee_profile(0, -0.500, -1)
    profile_points += reverse_ogee

    list_length = len(profile_points) - 1
    ogee_x = profile_points[list_length][0]
    ogee_y = profile_points[list_length][1]
    profile_points.append((ogee_x - 0.1875, ogee_y))
    profile_points.append((ogee_x - 0.1875, ogee_y + 0.25))
    profile_points.append((0,ogee_y + 0.25))
    profile_points.append((0,ogee_y + 0.50))
    
    # Create the 2D profile
    profile = cq.Workplane("XZ").polyline(profile_points).close().extrude(length)

    return profile
    


    

In [14]:
def window_frame(center_x, center_y, center_z):

    # Parameters
    frame_depth = 4
    frame_width = 5
    sill_inside_height = 5
    sill_outside_height = 3
    sill_width = 5
    bead_size = 0.625
    tenon_size = 2
    header_length = (frame_width * 2) + rail_length
    lap_thickness = 1
    lap_size = 3

    # Window frame
    window_frame = cq.Assembly()

    # Left window frame
    left_x = center_x - (header_length/2)
    left_z = center_z + (pulley_stile_length/2) - (frame_width/2) + 1.5
    left_frame = beaded_board(frame_width, frame_depth, bead_size).extrude(pulley_stile_length + 5).rotate((center_x, center_y, center_z),(1,0,0),90).translate((left_x, center_y, left_z))
    left_tenon_a = left_frame.faces(">Z").workplane().center(left_x + (frame_width/2), center_y + tenon_size).rect(tenon_size,tenon_size,forConstruction=True).wires().toPending().extrude(frame_width)
    left_tenon_b = left_tenon_a.faces("<Z").workplane().rect(tenon_size,tenon_size,forConstruction=True).wires().toPending().extrude(frame_width-2)
    window_frame.add(left_tenon_b, color=sash_color)
    
    # Right window frame
    right_x = center_x + (header_length/2) - frame_width
    right_z = center_z + (pulley_stile_length/2) - (frame_width/2) + 1.5
    right_frame = beaded_board(frame_width, frame_depth, bead_size).extrude(pulley_stile_length + 5).rotate((center_x, center_y, center_z),(1,0,0),90).translate((right_x, center_y, right_z))
    right_tenon_a = right_frame.faces(">Z").workplane().center(right_x + (frame_width/2), center_y + tenon_size).rect(tenon_size,tenon_size,forConstruction=True).wires().toPending().extrude(frame_width)
    right_tenon_b = right_tenon_a.faces("<Z").workplane().rect(tenon_size,tenon_size,forConstruction=True).wires().toPending().extrude(frame_width-2)
    window_frame.add(right_tenon_b, color=sash_color)
    
    # Top window frame
    top_z = center_z + ((pulley_stile_length + frame_width)/2) + 1.5
    top_x = center_x - (header_length/2)
    top_frame = beaded_board(frame_depth, frame_width, bead_size).extrude(header_length).rotate((center_x, center_y, center_z),(0,0,1),90).translate((top_x, center_y, top_z))
    top_mortise_a = top_frame.faces("<Z").workplane().center(right_x + (frame_width/2), center_y + tenon_size).rect(tenon_size,tenon_size,forConstruction=True).wires().toPending().cutBlind(frame_width)
    top_mortise_b = top_mortise_a.faces("<Z").workplane().center(left_x + (frame_width/2), center_y + tenon_size).rect(tenon_size,tenon_size,forConstruction=True).wires().toPending().cutBlind(frame_width)
    top_tenon_a = top_mortise_b.faces("<X").workplane().center(1, center_z + lap_size/2 + 1).rect(lap_thickness, frame_depth, forConstruction=True).wires().toPending().extrude(frame_depth)
    top_tenon_b = top_tenon_a.faces(">X").workplane().rect(lap_thickness, frame_depth, forConstruction=True).wires().toPending().extrude(frame_depth)
    window_frame.add(top_tenon_b, color=sash_color)
    
    # Bottom window frame (sill)
    bottom_x = center_x - (header_length/2)
    bottom_z = center_z - ((pulley_stile_length + frame_width)/2) - 0.5
    bottom_frame = beaded_sill(sill_width, sill_inside_height, sill_outside_height, bead_size).extrude(header_length).rotate((center_x, center_y, center_z),(0,0,1),90).translate((bottom_x, center_y, bottom_z))
    bottom_mortise_a = bottom_frame.faces("<Z").workplane().center(right_x + (frame_width/2), center_y + tenon_size).rect(tenon_size,tenon_size,forConstruction=True).wires().toPending().cutBlind(frame_width)
    bottom_mortise_b = bottom_mortise_a.faces("<Z").workplane().center(left_x + (frame_width/2), center_y + tenon_size).rect(tenon_size,tenon_size,forConstruction=True).wires().toPending().cutBlind(frame_width)
    bottom_tenon_a = bottom_mortise_b.faces("<X").workplane().center(-7, center_z + lap_size/2).rect(lap_thickness,lap_size,forConstruction=True).wires().toPending().extrude(frame_depth)
    bottom_tenon_b = bottom_tenon_a.faces(">X").workplane().rect(lap_thickness,lap_size,forConstruction=True).wires().toPending().extrude(frame_depth)
    window_frame.add(bottom_tenon_b, color=sash_color)

    return window_frame

In [27]:
def top_sash(center_x, center_y, center_z):

    sash = cq.Assembly()

    # Parameters
    stile_z = center_z + top_stile_length - thickness
    stile_y = center_y + (thickness/2) + thickness + 0.75

    # Left Stile
    left_x = center_x - (rail_length/2)
    left_stile = add_stile(top_stile_length).rotate((center_x, center_y, center_z),(1,0,0),90).rotate((center_x, center_y, center_z),(0,0,1),90).translate((left_x, stile_y, stile_z))
    sash.add(left_stile, name = "top_left_stile", color=sash_color)

    # Right Stile
    right_x = center_x + (rail_length/2)
    right_stile = add_stile(top_stile_length).rotate((center_x, center_y, center_z),(1,0,0),270).rotate((center_x, center_y, center_z),(0,0,1),90).translate((right_x, stile_y, stile_z - top_stile_length))
    sash.add(right_stile, name = "top_right_stile", color=sash_color)

    # Top Rail
    top_rail_x = center_x + (rail_length/2)
    top_rail = add_rail(top_rail_width, rail_length).rotate((center_x, center_y, center_z),(0,0,1),90).rotate((center_x, center_y, center_z),(0,1,0),180).translate((top_rail_x, stile_y, stile_z))
    sash.add(top_rail, name = "top_rail", color=sash_color)

    # Meeting Rail
    meeting_rail_x = center_x + (rail_length/2)
    meeting_rail = add_top_meeting_rail(rail_length).rotate((center_x, center_y, center_z),(0,0,1),90).rotate((center_x, center_y, center_z),(0,1,0),180).rotate((center_x, center_y, center_z),(0,0,1),180).translate((-meeting_rail_x, stile_y , -stile_z + top_stile_length - thickness))
    sash.add(meeting_rail, name = "top_meeting_rail", color=sash_color)

    # Vertical muntins
    vertical_muntin_left_x = left_x + stile_width + light_width - 0.25 + (muntin_width/2)
    vertical_muntin_right_x = right_x - stile_width - light_width + 0.25
    top_left_vertical_muntin = ogee_muntin(top_stile_length).rotate((center_x, center_y, center_z),(1,0,0),90).rotate((center_x, center_y, center_z),(0,0,1),90).translate((vertical_muntin_left_x, stile_y - thickness, stile_z))
    top_right_vertical_muntin = ogee_muntin(top_stile_length).rotate((center_x, center_y, center_z),(1,0,0),90).rotate((center_x, center_y, center_z),(0,0,1),90).translate((vertical_muntin_right_x, stile_y - thickness, stile_z))
    sash.add(top_left_vertical_muntin, name = "top_left_vertical_muntin", color=sash_color)
    sash.add(top_right_vertical_muntin, name = "top_right_vertical_muntin", color=sash_color)

    # Horizontal muntins
    if top_sash_lights == 6:
        muntins = 1
    elif top_sash_lights == 9:
        muntins = 2
    else:
        muntins = 0

    for muntin in range(1,muntins + 1):

        horizontal_muntin_z = stile_z - stile_width - (muntin * (light_height + 0.25 - (muntin_width/2)))
        horizontal_muntin = ogee_muntin(rail_length).rotate((center_x, center_y, center_z),(0,0,1),90).translate((top_rail_x - rail_length, stile_y - thickness, horizontal_muntin_z))
        sash.add(horizontal_muntin, name=f"top_horizontal_muntin_{muntin}", color=sash_color)
    
    return sash
    

In [44]:
def bottom_sash(center_x, center_y, center_z):

    sash = cq.Assembly()

    #Parameters
    stile_z = center_z - bottom_stile_length + (thickness/2) + top_stile_length - thickness
    stile_y = center_y + (thickness/2) + 0.375

    # Left stile
    left_x = center_x - (rail_length/2) + stile_width
    left_stile = add_stile(bottom_stile_length).rotate((center_x, center_y, center_z),(1,0,0),90).rotate((center_x, center_y, center_z),(0,0,1),90).rotate((0,0,0),(0,1,0),180).translate((left_x,stile_y, stile_z - bottom_stile_length))
    sash.add(left_stile, name = "bottom_left_stile", color=sash_color)

    # Right stile
    right_x = center_x + (rail_length/2) - stile_width
    right_stile = add_stile(bottom_stile_length).rotate((center_x, center_y, center_z),(1,0,0),270).rotate((center_x, center_y, center_z),(0,0,1),90).rotate((0,0,0),(0,1,0),180).translate((right_x, stile_y, stile_z))
    sash.add(right_stile, name = "bottom_right_stile", color=sash_color)

    # Bottom rail
    bottom_rail_x = center_x - (rail_length/2)
    bottom_rail = add_rail(bottom_rail_width, rail_length).rotate((center_x, center_y, center_z),(0,0,1),90).rotate((center_x, center_y, center_z),(0,1,0),180).rotate((0,0,0),(0,1,0),180).translate((bottom_rail_x, stile_y, stile_z - bottom_stile_length))
    sash.add(bottom_rail, name = "bottom_rail", color=sash_color)

    # Meeting Rail
    meeting_rail_x = center_x + (rail_length/2)
    meeting_rail = add_bottom_meeting_rail(rail_length).rotate((center_x, center_y, center_z),(0,0,1),90).rotate((center_x, center_y, center_z),(1,0,0),180).rotate((0,0,0),(0,1,0),180).rotate((0,0,0),(0,0,1),180).translate((-meeting_rail_x, -stile_y + 0.6, stile_z))
    sash.add(meeting_rail, name = "bottom_meeting_rail", color=sash_color)

    # Vertical muntins
    vertical_muntin_left_x = left_x + light_width - 0.25 + (muntin_width/2)
    vertical_muntin_right_x = right_x - light_width + 0.25
    bottom_left_vertical_muntin = ogee_muntin(bottom_stile_length).rotate((center_x, center_y, center_z),(1,0,0),90).rotate((center_x, center_y, center_z),(0,0,1),90).translate((vertical_muntin_left_x, stile_y - thickness, stile_z))
    bottom_right_vertical_muntin = ogee_muntin(bottom_stile_length).rotate((center_x, center_y, center_z),(1,0,0),90).rotate((center_x, center_y, center_z),(0,0,1),90).translate((vertical_muntin_right_x, stile_y - thickness, stile_z))
    sash.add(bottom_left_vertical_muntin, name = "bottom_left_vertical_muntin", color=sash_color)
    sash.add(bottom_right_vertical_muntin, name = "bottom_right_vertical_muntin", color=sash_color)

    # Horizontal muntins
    if bottom_sash_lights == 6:
        muntins = 1
    elif bottom_sash_lights == 9:
        muntins = 2
    else:
        muntins = 0

    for muntin in range(1,muntins + 1):
        horizontal_muntin_z = stile_z - (muntin * (light_height - 0.25 + (muntin_width/2)))
        horizontal_muntin = ogee_muntin(rail_length).rotate((center_x, center_y, center_z),(0,0,1),90).translate((bottom_rail_x, stile_y - thickness, horizontal_muntin_z))
        sash.add(horizontal_muntin, name=f"bottom_horizontal_muntin_{muntin}", color=sash_color)

    return sash

In [45]:
frame = window_frame(0,0,0)
bottom_sash_a = bottom_sash(0,0,0)
top_sash_a = top_sash(0,0,0)
show(frame, top_sash_a, bottom_sash_a)
#bmr = add_top_meeting_rail(100)
#show(bmr)


cccccccccccccc+ccc
